In [ ]:
import requests, json, re
from collections import Counter
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [3]:
url = 'https://www.gutenberg.org/files/11/11-0.txt'

response = requests.get(url)
response.encoding = 'utf-8'

word = re.findall(r'\b[a-z]+\b', response.text.lower())
count = Counter(word)
print(count.most_common(10))

[('the', 1653), ('and', 873), ('to', 729), ('a', 637), ('it', 592), ('she', 549), ('i', 524), ('of', 517), ('said', 460), ('alice', 398)]


In [ ]:
# Scraping Quotes
base_url = 'https://quotes.toscrape.com'
url = base_url

data = []

while url:
    print(f'Scrape to: {url}')
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    quotes = soup.find_all('div', class_='quote')
    for quote in quotes:
        text = quote.find('span', class_='text').string
        author = quote.find('small', class_='author').string

        tag_parent = quote.find('div', class_='tags')
        tags = tag_parent.find('meta', class_='keywords').get('content')
        tags_split = [x.capitalize() for x in tags.strip().split(',')]
        data.append({
            'author': author,
            'quote': text,
            'tags': tags_split
        })
    next_page = soup.find('li', class_='next')
    # Pagination
    if next_page:
        next_url = next_page.find('a')['href']
        url = base_url + next_url
    else:
        url = None

with open('quotes.json', 'w', encoding='utf-8') as file:
    save = json.dump(data, file, indent=4)

Scrape to: https://quotes.toscrape.com
Scrape to: https://quotes.toscrape.com/page/2/
Scrape to: https://quotes.toscrape.com/page/3/
Scrape to: https://quotes.toscrape.com/page/4/
Scrape to: https://quotes.toscrape.com/page/5/
Scrape to: https://quotes.toscrape.com/page/6/
Scrape to: https://quotes.toscrape.com/page/7/
Scrape to: https://quotes.toscrape.com/page/8/
Scrape to: https://quotes.toscrape.com/page/9/
Scrape to: https://quotes.toscrape.com/page/10/


In [15]:
# Scraping Books Information
base_url = 'https://books.toscrape.com'
url = base_url

map_rating = {
    'one': 1,
    'two': 2,
    'three': 3,
    'four': 4,
    'five': 5
}

data = []

while url:
    print(f'Scrape to: {url}')
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    books = soup.section.find_all('article', class_='product_pod')
    for book in books:
        title = book.find('div', class_='image_container').a.img.get('alt')
        picture_link = book.find('div', class_='image_container').a.img.get('src')
        rating_str = book.p.get('class')[1].lower()
        rating_book = map_rating[rating_str]
        product_price = book.find('div', class_='product_price')
        stock_availability = product_price.find('p', class_='instock availability').text.strip()

        catalogue = book.find('div', class_='image_container').a["href"]
        cat_url = urljoin(url, catalogue)
        detail_response = requests.get(cat_url)
        if detail_response.status_code != 200:
            print("Gagal:", cat_url)
            continue
        detail_book = BeautifulSoup(detail_response.content, 'html.parser')
        breadcrumb = detail_book.find('ul', class_='breadcrumb')
        category = breadcrumb.find_all('a')[2].get_text(strip=True)
        table = detail_book.find('table', class_='table table-striped')
        tr = table.find_all('tr')
        upc = tr[0].td.text
        product_type = tr[1].td.text
        price_exc_tax = tr[2].td.text
        price_inc_tax = tr[3].td.text
        tax = tr[4].td.text
        num_stock_pattern = re.search(r'\d+', tr[5].td.text)
        num_stock = num_stock_pattern.group()
        review = tr[6].td.text

        data.append({
                'title': title.capitalize(),
                'category': category.capitalize(),
                'product_type': product_type.capitalize(),
                'rating': int(rating_book),
                'upc': upc,
                'number_of_stock': int(num_stock),
                'stock_availability': stock_availability,
                'number_of_review': int(review),
                'price_exclude_tax': price_exc_tax,
                'price_include_tax': price_inc_tax,
                'tax': tax,
                'image_src': picture_link
        })

    next_page = soup.section.find_all('li', class_='next')
    if next_page:
        next_url = next_page[0].a.get('href')
        url = urljoin(url, next_url)
        print("Next URL:", url)
    else:
        url = None

with open('books_to_scrape.json', 'w', encoding='utf-8') as file:
    save = json.dump(data, file, indent=4)


Scrape to: https://books.toscrape.com
Next URL: https://books.toscrape.com/catalogue/page-2.html
Scrape to: https://books.toscrape.com/catalogue/page-2.html
Next URL: https://books.toscrape.com/catalogue/page-3.html
Scrape to: https://books.toscrape.com/catalogue/page-3.html
Next URL: https://books.toscrape.com/catalogue/page-4.html
Scrape to: https://books.toscrape.com/catalogue/page-4.html
Next URL: https://books.toscrape.com/catalogue/page-5.html
Scrape to: https://books.toscrape.com/catalogue/page-5.html
Next URL: https://books.toscrape.com/catalogue/page-6.html
Scrape to: https://books.toscrape.com/catalogue/page-6.html
Next URL: https://books.toscrape.com/catalogue/page-7.html
Scrape to: https://books.toscrape.com/catalogue/page-7.html
Next URL: https://books.toscrape.com/catalogue/page-8.html
Scrape to: https://books.toscrape.com/catalogue/page-8.html
Next URL: https://books.toscrape.com/catalogue/page-9.html
Scrape to: https://books.toscrape.com/catalogue/page-9.html
Next URL: 